# 43モード・密なThread用の圧縮ランクを先に検証する

直接法の123は改善しましたが、旧ランク100の提案場は圧縮段階で崩れました。保存済みの直接複素場と係数を基準に、CGH・伝搬を行う前にランクを検証します。直接J0の全生成を繰り返しません。

最大ランク512で共有基底を一度学習し、その同じ基底を100・160・256・384・512列に制限して比較します。元のランク100で個別に学習した辞書とは区別します。検証に使った親辞書と列を最終比較にもそのまま適用します。

基準は初期複素場の相対L2誤差1%以下、強度の相対L2誤差3%以下。これだけで伝搬後の画質を保証するものではありません。どのランクも通らなければ、結果を残して止めます。説明は `LSH数字123_圧縮失敗の診断.md`。


In [ ]:
import importlib
import json
from pathlib import Path
import lsh_digit1_theory as theory
import lsh_digit1_proposed as proposed
import lsh_digits123_rank_audit as rank_audit
theory = importlib.reload(theory)
proposed = importlib.reload(proposed)
rank_audit = importlib.reload(rank_audit)

REFERENCE_DIR = Path('lsh_digits123_improved_proposed_wide_dense_coupled_outputs_rank100_v1')
AUDIT_DIR = Path('lsh_digits123_rank_audit_outputs_v1')
RANKS = (100, 160, 256, 384, 512)
MAX_COMPLEX_ERROR = 0.01
MAX_INTENSITY_ERROR = 0.03
RUN_FINITE_VALIDATION = False  # Trueにすると合格後にCGH・伝搬を再検証
VALIDATION_VERSION = 'v1'
source = json.loads((REFERENCE_DIR/'summary.json').read_text(encoding='utf-8'))
config = theory.Digit1Config(**source['config'])
print('元の初期複素場誤差:', source['metrics']['initial_complex_proposed_vs_direct']['complex_relative_l2'])
print('元のシフト診断:', source['shift_diagnostics'])
print('候補ランク:', RANKS)
print('最大ランクの基底を一度準備します。必要なランクは実測で決めます。')


In [ ]:
from IPython.display import Image, display
audit = rank_audit.run_rank_audit(REFERENCE_DIR, AUDIT_DIR, RANKS,
    max_complex_error=MAX_COMPLEX_ERROR, max_intensity_error=MAX_INTENSITY_ERROR)
display(Image(filename=str(AUDIT_DIR/'rank_initial_error.png'), width=1000))
for row in [audit['original_rank_result'], *audit['cases']]:
    print(row)
if audit['accepted_rank'] is None:
    print('合格ランクなし。CGH・伝搬は実行しません。最大ランクの拡張や基底設計を検討します。')
    print('候補を変える場合はAUDIT_DIRもv2などへ変更してください。')
else:
    print('合格ランク:', audit['accepted_rank'])
    print('この親辞書を使う設定:', audit['recommendation'])
print('保存先:', AUDIT_DIR.resolve())


In [ ]:
# 任意の次段階。直接法は新しく生成し、計算時間をキャッシュで短縮しません。
validation_summary = None
if not RUN_FINITE_VALIDATION:
    print('初期場のランク検証までで終了。CGH・伝搬を確認する場合は上の設定をTrueにしてください。')
elif audit['recommendation'] is None:
    raise ValueError('合格ランクがないため伝搬へ進みません。')
else:
    rec = audit['recommendation']
    output_dir = Path(f"lsh_digits123_rank_verified_outputs_rank{rec['rank']}_parent{rec['library_parent_rank']}_{VALIDATION_VERSION}")
    if (output_dir/'summary.json').is_file() and (output_dir/'manifest.json').is_file():
        validation_summary = json.loads((output_dir/'summary.json').read_text(encoding='utf-8'))
        valid = (validation_summary.get('study') == 'digit1_direct_vs_projected_shared_core'
            and theory.config_payload(validation_summary['config']) == theory.config_payload(config)
            and validation_summary.get('rank') == rec['rank']
            and validation_summary.get('library_parent_rank') == rec['library_parent_rank']
            and validation_summary.get('seed') == source['seed']
            and validation_summary.get('library_path') == rec['library_path']
            and validation_summary.get('mode_spacing_factor', 1) == source.get('mode_spacing_factor', 1)
            and validation_summary.get('basis_strategy') == 'shift_aware')
        if not valid:
            raise ValueError('保存結果が別条件です。VALIDATION_VERSIONを変更してください。')
        err = validation_summary['metrics']['initial_complex_proposed_vs_direct']['complex_relative_l2']
        if err > MAX_COMPLEX_ERROR:
            raise ValueError('保存結果が初期場の基準を満たしていません。')
    elif output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError('途中結果があります。VALIDATION_VERSIONを変更してください。')
    else:
        validation_summary = proposed.run_comparison(output_dir, config,
            rank=rec['rank'], library_parent_rank=rec['library_parent_rank'],
            library_path=rec['library_path'], seed=source['seed'],
            mode_spacing_factor=source.get('mode_spacing_factor', 1), basis_strategy='shift_aware',
            max_initial_complex_error=MAX_COMPLEX_ERROR, run_shift_diagnostics=True)
    print('有限開口/CGH結果:', output_dir.resolve())


In [ ]:
if validation_summary is not None:
    display(Image(filename=str(output_dir/'target_direct_proposed.png'), width=1200))
    display(Image(filename=str(output_dir/'cgh_sheet_comparison.png'), width=1200))
    print('提案法/直接法の再生強度:', validation_summary['metrics']['cgh_sheet_proposed_vs_direct'])
    print('直接法/目標:', validation_summary['metrics']['direct_cgh_sheet_vs_target'])
    print('提案法/目標:', validation_summary['metrics']['proposed_cgh_sheet_vs_target'])
    print('計算時間:', validation_summary['timings_seconds'])
else:
    print('伝搬像はまだ計算していません。初期場の診断結果のみです。')
